In [1]:
import warnings
import os 
from pathlib import Path

import pandas as pd
import scanpy as sc 
import numpy as np 
from dotenv import dotenv_values

warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

SC_FP = DATADIR / "9_objects" / "singlecell" / 'hypomap3d_singlecell_all.h5ad' # for descr mapper
SC_ST_FP = DATADIR / "9_objects" / "spatial" / 'hypomap3d_spatial_cell_level.h5ad' # for concatenated object
OPENST_RAW_FP = DATADIR / "7_spatial" / "raw" / "Bauder_spatial.h5ad"
HMv1_FP = "/cephfs/scratch/bruening_scratch/lsteuernagel/data/hypoMap_publication/hypoMap_anndata.h5ad"


# singlecell 

In [3]:
adata_sc = sc.read_h5ad(SC_FP)

### Cell numbers per atlas

Total

In [17]:
n_total = len(adata_sc)
n_total

1564064

New cells

In [19]:
n_new = len(adata_sc[adata_sc.obs.Dataset.str.contains("SC2") & ~adata_sc.obs.Author.str.contains("Yao")])
n_new

960001

Ratios of atlases

In [20]:
n_abc = len(adata_sc[adata_sc.obs.Author.str.contains("Yao")])
n_hmv1 = len(adata_sc[adata_sc.obs.Dataset.str.contains("SC1")])

print(f"HMv1: {n_hmv1 / n_total}, ABC: {n_abc / n_total}, NEW: {n_new / n_total}")

HMv1: 0.23932076948257872, ABC: 0.14689296601673588, NEW: 0.6137862645006854


### Number of clusters

In [5]:
for i in range(1,6):
    n_N = adata_sc.obs.loc[~adata_sc.obs.L1_ID.str.contains("NN"), f"L{i}_ID"].nunique()
    n_NN = adata_sc.obs.loc[adata_sc.obs.L1_ID.str.contains("NN"), f"L{i}_ID"].nunique()
    n_both = n_N + n_NN
    print(f"L{i}: {n_both} cluters (N: {n_N}, NN: {n_NN})")

L1: 6 cluters (N: 2, NN: 4)
L2: 82 cluters (N: 68, NN: 14)
L3: 274 cluters (N: 246, NN: 28)
L4: 716 cluters (N: 670, NN: 46)
L5: 932 cluters (N: 886, NN: 46)


### Number of L4 clusters splitting on L5

In [38]:
n_l5_per_l4 = adata_sc.obs.groupby("L4_ID")["L5_ID"].nunique()

n_split = (n_l5_per_l4 > 1).sum()

n_split

np.int64(185)

### Number of cells in new Bauder dataset

In [4]:
bauder = adata_sc[adata_sc.obs.Author.str.contains("Bauder")]

In [5]:
len(bauder)

28611

### Number of cell types in BauderC

In [6]:
bauder.obs.L2_descr = bauder.obs.L2_descr.astype(str)
bauder.obs.L4_descr = bauder.obs.L4_descr.astype(str)

/REDACTED-path/HypoMap3D/.venv/lib/python3.11/site-packages/pandas/core/generic.py:6351: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  self[name] = value


In [12]:
bauder.obs.loc[(~bauder.obs.L2_descr.str.contains("NN")), "L2_descr"].nunique() / adata_sc.obs.loc[(~adata_sc.obs.L2_descr.str.contains("NN")), "L2_descr"].nunique()

0.9852941176470589

In [10]:
bauder.obs.loc[(~bauder.obs.L2_descr.str.contains("NN")), "L4_descr"].nunique()

662

### Gipr expressing clusters (control only)

In [9]:
gipr = adata_sc[adata_sc.obs.IsControl].copy()
sc.pp.normalize_total(gipr, target_sum=10_000)
gipr.obs["expresses_Gipr"] = gipr[:, "Gipr"].X.toarray() > 0

In [10]:
gipr.obs.loc[:, ["L3_descr", "expresses_Gipr"]].groupby("L3_descr")["expresses_Gipr"].mean().sort_values(ascending=False).head(15)

L3_descr
Gl3-32 PH Pthlh      0.056247
NN3-16 Peri_1        0.043959
NN3-2 Oligo_2_1      0.043434
Gl3-31 PH Crhbp      0.031536
Gb3-146 DMH Glp1r    0.027999
NN3-1 Oligo_1_1      0.027957
Gl3-67 PH Npas1      0.019608
Gl3-8 PSTN Egfem1    0.019341
NN3-19 VLMC_3        0.019168
Gb3-232 TMv Hdc      0.018724
Gl3-51 LHA Foxa1     0.018182
Gb3-220 ARH Tbx19    0.017778
Gl3-61 LHA Robo3     0.017751
Gl3-24 DMH Sst       0.017686
Gl3-91 PMv Sox6      0.017501
Name: expresses_Gipr, dtype: float64

In [11]:
gipr.obs.loc[:, ["L4_descr", "expresses_Gipr"]].groupby("L4_descr")["expresses_Gipr"].mean().sort_values(ascending=False).head(15)

L4_descr
Gl4-80 PH Pthlh.Gipr         0.309764
Gl4-76 PH Crhbp.Calca        0.076807
Gb4-391 DMH Glp1r.Tbx19      0.073656
Gl4-145 MPN Lhx9.Col24a1     0.060317
Gb4-508 CP Ndst3.Col5a2      0.051181
NN4-25 Peri_1-1              0.043959
NN4-3 Oligo_2_1-1            0.043434
Gl4-18 PSTN Egfem1.Maf       0.036000
Gb4-612 ARH Tbx19.Anxa2      0.033473
NN4-2 Oligo_1_1-2            0.032763
Gl4-154 LHA Rassf4.Igfbp3    0.030973
Gb4-609 AHN Qrfprl.Scube2    0.028668
Gl4-9 PSTN Irx5.Crispld2     0.027933
Gl4-28 VPM Pitx2.Lhfpl1      0.027778
NN4-1 Oligo_1_1-1            0.027677
Name: expresses_Gipr, dtype: float64

So, the only cell types expressing Gipr > 5% are 

Gl4-80 PH Pthlh.Gipr         0.309764
Gl4-76 PH Crhbp.Calca        0.076807
Gb4-391 DMH Glp1r.Tbx19      0.073656
Gl4-145 MPN Lhx9.Col24a1     0.060317
Gb4-508 CP Ndst3.Col5a2      0.051181

--> Notably, we detected expression of the glucose-dependent insulinotropic polypeptide receptor (Gipr), another target of incretin-based drugs at low levels in Gb4-391 (7.4%). In HypoMap3D this is the only neuronal cell type besides an anterior (Gl4-145 MPN Lhx9.Col241, 6%) and two posterior hypothalamic cell types (Gl4-80 PH Pthlh.Gipr 31%, Gl4-76 PH Crhbp.Calca 7.7%) that expresses Gipr in more than 5% of cells 

In [12]:
gipr.obs.loc[:, ["L4_descr", "expresses_Gipr"]].groupby("L4_descr")["expresses_Gipr"].sum().sort_values(ascending=False).head(15)

L4_descr
NN4-1 Oligo_1_1-1           2604
NN4-2 Oligo_1_1-2            180
NN4-25 Peri_1-1              163
Gb4-391 DMH Glp1r.Tbx19      111
Gl4-80 PH Pthlh.Gipr          92
NN4-3 Oligo_2_1-1             85
NN4-29 VLMC_3-1               54
Gl4-76 PH Crhbp.Calca         51
NN4-34 Astro_3-1              44
Gl4-217 PH Fgf10              43
Gl4-38 PMd Ctxn3.Chst9        41
Gb4-612 ARH Tbx19.Anxa2       40
Gl4-41 MMme Kitl.Nid1         40
Gl4-145 MPN Lhx9.Col24a1      38
Gl4-32 VPM Pitx2.Scn4b        33
Name: expresses_Gipr, dtype: int64

### DMH Glp1r L4 clusters detection 

In [8]:
result = (
    adata_sc.obs.loc[
        adata_sc.obs.L4_descr.str.contains("DMH Glp1r"),
        ["L4_descr", "Author"]
    ]
    .groupby("L4_descr")["Author"]
    .nunique()
    .reset_index(name="n_authors")
)
result.loc[result.n_authors > 0]

,L4_descr,n_authors
69,Gb4-388 DMH Glp1r.Slc26a7,32
70,Gb4-389 DMH Glp1r.Erg,33
71,Gb4-390 DMH Glp1r.Ghrh,36
72,Gb4-391 DMH Glp1r.Tbx19,33
73,Gb4-392 DMH Glp1r.Nfix,31


### Counts / genes per cell

In [23]:
hmv1 = sc.read_h5ad(HMv1_FP)

In [27]:
qc_df_v1 = sc.pp.calculate_qc_metrics(hmv1, log1p =False, inplace=False)

In [28]:
qc_df_v1[0].mean(axis=0)

n_genes_by_counts              2171.826540
total_counts                   3010.694057
pct_counts_in_top_50_genes        6.183843
pct_counts_in_top_100_genes      11.009382
pct_counts_in_top_200_genes      19.397376
pct_counts_in_top_500_genes      40.046021
dtype: float64

In [24]:
adata_sc.obs["total_counts"].mean(), adata_sc.obs.n_genes.mean()

(np.float64(7132.473906438611), np.float64(2830.325556371095))

### Neuronal cell types vs HMv1

In [51]:
num_n_l4 = adata_sc.obs.loc[~adata_sc.obs.L4_ID.str.contains("NN"), "L4_ID"].nunique()

num_n_c465 = hmv1.obs.loc[hmv1.obs["C465_named"].astype(str).str.contains("GABA|GLU", regex=True), "C465_named"].nunique()

In [52]:
print(f"{num_n_l4 - num_n_c465} more neruonal clusters on L4 than on C465")

350 more neruonal clusters on L4 than on C465


### NN

Cell count all

In [41]:
len(adata_sc.obs.loc[adata_sc.obs.L1_ID.str.contains("NN")])

520768

In [42]:
adata_sc.obs.L1_descr.value_counts()

L1_descr
Gb1-2 GABA          562974
Gl1-1 Glut          480322
NN1-1 OPC-Oligo     231581
NN1-4 Astro-Epen    207489
NN1-3 Vascular       43245
NN1-2 Immune         38453
Name: count, dtype: int64

Cell count in control only

In [40]:
len(adata_sc.obs.loc[adata_sc.obs.IsControl & adata_sc.obs.L1_ID.str.contains("NN")])

289866

In [43]:
adata_sc.obs.loc[adata_sc.obs.IsControl, "L1_descr"].value_counts()

L1_descr
Gb1-2 GABA          328018
Gl1-1 Glut          279075
NN1-1 OPC-Oligo     140839
NN1-4 Astro-Epen    108835
NN1-3 Vascular       22077
NN1-2 Immune         18115
Name: count, dtype: int64

Number of Endo cells 

In [44]:
len(adata_sc[adata_sc.obs.L2_ID.str.contains("NN") & adata_sc.obs.L3_descr.str.contains("Endo") & adata_sc.obs.IsControl])

10328

In [45]:
len(adata_sc[adata_sc.obs.L2_ID.str.contains("NN") & adata_sc.obs.L3_descr.str.contains("Oligo") & adata_sc.obs.IsControl])

107738

### Sex

Cell counts

In [46]:
for sex in ["male", "female"]:
    n_all = np.sum(adata_sc.obs.sex_per_sample_inferred == sex) 
    n_ctrl = np.sum((adata_sc.obs.sex_per_sample_inferred == sex) & (adata_sc.obs.IsControl)) 
    print(f"{sex}: all: {n_all}, control only: {n_ctrl}")


male: all: 851810, control only: 436654
female: all: 581726, control only: 402559


Number of datasets with both sexes

In [48]:

# For each Dataset, check if it contains both sexes
datasets_with_both = (
    adata_sc[adata_sc.obs.sex_per_sample_inferred.isin(["male", "female"])].obs.groupby("Dataset")["sex_per_sample_inferred"]
      .nunique()                      # count unique sexes
      .eq(2)                          # TRUE for datasets with 2 unique sexes (male+female)
)

# Count how many datasets have both sexes
count = datasets_with_both.sum()

print("Number of datasets with both female and male:", count)


Number of datasets with both female and male: 15


# Spatial

## OpenST

In [30]:
openst_raw = sc.read_h5ad(OPENST_RAW_FP, backed="r")

### x / Bregma range

In [36]:
x_min_openst = round(openst_raw.obs.ccf_x.min() *1000, 0)
x_max_openst = round(openst_raw.obs.ccf_x.max() * 1000, 0)
x_min_openst, x_max_openst

(np.float64(6018.0), np.float64(7803.0))

There is [discussion](https://community.brain-map.org/t/how-to-transform-ccf-x-y-z-coordinates-into-stereotactic-coordinates/1858/13) on how to align Bregma to CCF as "there is no ground truth conversion between CCF and Bregma coordinates" as GTMeijer pointed out. 

Hence, as to get a very rough estimate for the publication, we can use the quite simplified 

X coord in microns = (X coord in CCF - 5400) * -1

In [37]:
for x in [x_min_openst, x_max_openst]:
    print((x-5400)*-1)

-618.0
-2403.0


### Number of cells per slice pre filtering 

In [34]:
openst_raw.obs.groupby("ccf_x")["ccf_x"].size().mean()

np.float64(31588.63157894737)

### Mean total UMI count pre filtering 

In [35]:
openst_raw.obs["total_counts"].mean()

np.float64(354.17265038721456)